# brier - M6.5: cross-family benchmark (raw / L0 / L1 / L2 on banking20)

The same protocol, data, split and seed as the Qwen3-1.7B results in `docs/results.md`
(2,603 test items, 300 calibration labels, 500 unlabelled pool items), on six more models from
different families: Phi-4-mini, Mistral-7B-v0.3, SmolLM3-3B, OLMoE-1B-7B (mixture of
experts), LFM2-1.2B (hybrid convolution) and Falcon3-1B-**Base** (no chat template: plain-text
prompt, ADR-0008). L2 uses the 300-label budget only.

**Runtime:** A100, about 3 hours in total (Mistral-7B is the longest, about an hour).
Keep the tab open. Each model's results are saved as soon as it finishes, and a re-run skips
models that already have results, so after a disconnect just run the cells again. To keep
results across a runtime reset, set `SAVE_TO_DRIVE = True`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRIER_COMMIT = "355f90d46fef1ee36f8510b1c8e5ab2d46b34fad"  # main after PR #27 (M6.4)
DTYPE = "bfloat16"
L2_BUDGETS = "300"
SAVE_TO_DRIVE = False  # True: results go to Google Drive and survive a runtime reset

# key: (model id, pinned revision, batch size)
MODELS = {
    "falcon3base": ("tiiuae/Falcon3-1B-Base", "cb37ef3559b157b5c9d9226296ba01a5162da1f7", 64),
    "lfm2": ("LiquidAI/LFM2-1.2B", "40f3da0d0164913923aee9462c23077868b816a3", 64),
    "smollm3": ("HuggingFaceTB/SmolLM3-3B", "a07cc9a04f16550a088caea529712d1d335b0ac1", 64),
    "phi4mini": ("microsoft/Phi-4-mini-instruct", "cfbefacb99257ffa30c83adab238a50856ac3083", 64),
    "olmoe": ("allenai/OLMoE-1B-7B-0125-Instruct", "b89a7c4bc24fb9e55ce2543c9458ce0ca5c4650e", 32),
    "mistral7b": (
        "mistralai/Mistral-7B-Instruct-v0.3",
        "c170c708c41dac9275d15a8fff4eca08d52bab71",
        32,
    ),
}

In [ ]:
!rm -rf /content/brier /content/brier_repo  # leftovers from an earlier run
!git clone -q https://github.com/mohdUwaish59/brier.git /content/brier_repo
!git -C /content/brier_repo checkout -q {BRIER_COMMIT}
!pip install -q -e "/content/brier_repo[hf]"

In [ ]:
OUT_ROOT = "/content/results_m65"
if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_ROOT = "/content/drive/MyDrive/brier_m65"
print("results go to", OUT_ROOT)

In [ ]:
import glob
import os
import shutil
import subprocess
import sys
import time

# Every brier call runs in a fresh process (this notebook's own Python never imports brier).
for key, (model_id, revision, batch) in MODELS.items():
    out = f"{OUT_ROOT}/{key}"
    if glob.glob(f"{out}/*.json"):
        print(f"== {key}: results exist, skipping")
        continue
    print(f"\n===== {key}: {model_id} @ {revision[:7]} =====", flush=True)
    cmd = [
        sys.executable,
        "-m",
        "brier.bench",
        "run",
        "--model",
        model_id,
        "--revision",
        revision,
        "--dtype",
        DTYPE,
        "--batch-size",
        str(batch),
        "--levels",
        "raw,L0,L1,L2",
        "--l2-budgets",
        L2_BUDGETS,
        "--out",
        out,
    ]
    print("started", time.strftime("%H:%M:%S"), "- no output until this model finishes", flush=True)
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd="/content/brier_repo")  # noqa: S603
    print("finished", time.strftime("%H:%M:%S"))
    print(proc.stdout[-3000:])
    if proc.returncode != 0:
        print(f"!! {key} failed (exit {proc.returncode}):\n{proc.stderr[-3000:]}")
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)

In [ ]:
import glob
import json


def cell(res, level, metric):
    v = res["levels"].get(level, {}).get(metric)
    return f"{v[0]:.3f}" if v else "-"


rows = []
for key in MODELS:
    files_ = glob.glob(f"{OUT_ROOT}/{key}/*.json")
    if not files_:
        print(f"{key}: no results")
        continue
    with open(files_[0]) as f:
        rows.append((key, json.load(f)))

for metric in ("accuracy", "ece", "nll", "flip_rate"):
    print(f"\n{metric}")
    print("| model | raw | L0 | L1 | L2 (300) |")
    print("|---|---|---|---|---|")
    for key, res in rows:
        print(
            f"| {key} | "
            + " | ".join(cell(res, lv, metric) for lv in ("raw", "L0", "L1", "L2"))
            + " |"
        )
print()
for key, res in rows:
    e = res.get("l2_curve", {}).get(L2_BUDGETS.split(",")[-1], {})
    print(
        key,
        "L2 head:",
        {k: e.get(k) for k in ("layer", "solver", "alpha", "temperature")},
        "| wall s:",
        res["timing"]["wall_seconds"],
    )

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("/content/brier_m65_results", "zip", OUT_ROOT)
files.download("/content/brier_m65_results.zip")